# 세션 0 — 환경 설정 & 점검

이 노트북은 두 가지 역할을 합니다.

1. **환경 설정 가이드** — 강의에 필요한 Python · Poetry · VS Code · API 키를 준비하는 절차(아래 STEP 1~5).
2. **스모크 테스트** — 강의에서 가장 먼저 막히는 세 가지(라이브러리 · 모델 · API 키)를 미리 확인하는 코드 셀(맨 아래 "설치 확인").

> **사전 안내대로 미리 준비를 마친 분**은 곧장 아래 **"설치 확인"** 셀들을 위에서부터 실행해 보세요. 전부 통과하면 준비 끝입니다.
> 막힌 부분이 있으면 STEP 1~5와 맨 끝 **FAQ**를 참고하세요.

---

## 0. 사전 요건 & 준비물

**필수 사전 요건**
- Python 기본 문법 (함수, 클래스, dict/list)
- pip 으로 패키지 설치 경험
- LLM API 한 번이라도 호출해본 경험


**준비물 / 환경**

| 항목 | 내용 |
|------|------|
| 노트북 | 실습 진행, **개인 노트북 지참 필수** (오프라인 강의) |
| 실습 환경 | Visual Studio Code |
| Python | 3.12 또는 3.13 |
| 패키지 관리 | **Poetry** (강의 시작 전 사전 설치 권장) |
| OpenAI API 키 | 사전 결제 등록 필요 (**$10 이하** 사용 예정) |
| 디스크 | 최소 **10GB** 여유 공간 (bge-m3 · reranker · docling 모델 다운로드용) |

> **프로젝트 폴더 경로에 한글·공백이 없게** 하세요.
> 좋은 예 `~/lectures/bcmd_rag` · 나쁜 예 `~/내 문서/강의 자료/`

> **설치 중 막히면 AI에게 물어보세요.** 오류 메시지를 그대로 복사해 Claude · ChatGPT · Gemini 에 "Mac/Windows에서 어떤 단계 중 이 오류가 났다"고 붙여넣으면 대부분 단계별로 해결해 줍니다.

## 환경 설정 STEP 1~5

> 명령어는 **Mac = 터미널**, **Windows = PowerShell** 에서 실행합니다.

### STEP 1. Python 확인 (3.12 또는 3.13)
이미 설치돼 있으면 버전만 확인하면 됩니다.
```bash
python3 --version   # Mac
python --version    # Windows
# → Python 3.12.x 또는 3.13.x 가 나오면 OK
```
없거나 버전이 낮으면 <https://www.python.org/downloads/> 에서 3.12/3.13 설치.
(Windows는 설치 첫 화면에서 **Add Python to PATH** 체크 필수)

### STEP 2. Poetry 설치
이 강의는 **Poetry**로 라이브러리 버전을 강사와 동일하게 맞춥니다.
```bash
# Mac
python3 -m pip install --user pipx
python3 -m pipx ensurepath
pipx install poetry

# Windows (PowerShell) — pipx 설치 후 VS Code/PowerShell 완전 종료 → 재실행 → Poetry 설치
python -m pip install --user pipx
python -m pipx ensurepath
pipx install poetry
```
확인: `poetry --version` → `Poetry (version 2.x.x)` 나오면 성공.
(`command not found` 면 터미널을 새로 열고 다시 시도)

### STEP 3. 프로젝트 폴더로 이동 후 패키지 설치
압축 해제한 강의 폴더(`pyproject.toml`·`poetry.lock` 이 있는 곳)로 이동:
```bash
cd ~/lectures/bcmd_rag      # Mac (본인 경로로)
cd C:\lectures\bcmd_rag     # Windows (본인 경로로)

poetry install
```
> 패키지가 많아 **수~수십 분** 걸릴 수 있습니다(특히 torch 계열). 정상이니 기다리세요.

### STEP 4. OpenAI API 키 → `.env` 파일
프로젝트 폴더 안에 `.env` 파일을 만들고 키를 넣습니다.
```bash
# Mac
echo 'OPENAI_API_KEY=여기에_본인_키_붙여넣기' > .env
```
Windows는 메모장에 `OPENAI_API_KEY=...` 한 줄을 적고, 저장 시 **파일명 `.env` / 형식 "모든 파일"** 로 프로젝트 폴더에 저장.
- 키 발급: <https://platform.openai.com/> → API Keys → Create new secret key (키는 **한 번만** 표시되니 바로 복사)
- `.env` 는 **절대 공유·업로드 금지** (키 노출 시 요금 청구)

### STEP 5. VS Code에서 Poetry 가상환경을 Jupyter 커널로 등록
1. VS Code 확장 설치: **Python** · **Jupyter** · **Claude Code** (모두 검색해서 설치)
2. `File → Open Folder` 로 강의 폴더 열기
3. 가상환경을 커널로 등록(한 번만):
```bash
poetry run python -m ipykernel install --user --name=bcmd_rag_lecture --display-name "Python (bcmd_rag_lecture)"
```
4. 이 `.ipynb` 를 연 뒤 오른쪽 위 **Select Kernel → Python (bcmd_rag_lecture)** 선택
   - 목록에 안 보이면 3번을 다시 실행하고 VS Code 재시작

---

## 설치 확인 — 아래 셀들을 위에서부터 실행

여기까지 끝났으면 아래 코드 셀을 차례로 실행(`Shift+Enter`)하세요.
**키 → 라이브러리 → 모델 → LLM 호출 → 캐시** 순으로 점검합니다.
처음 모델을 받는 셀은 수 GB 다운로드로 시간이 걸릴 수 있습니다.

In [2]:
# 준비 셀 — 경로와 API 키를 읽어 온다
import os, time, pickle
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()                            # .env 파일에서 OPENAI_API_KEY 를 읽어 환경변수로 등록

ROOT = Path.cwd().parent                 # notebooks/ 의 상위 = 프로젝트 폴더
PDF = ROOT / "data" / "저출생_반전을_위한_대책_관계부처_합동_.pdf"   # 오늘 하루 쓸 유일한 문서
CACHE = ROOT / "cache"                   # 미리 계산해 둔 결과(청크·인덱스)가 담긴 폴더
CACHE.mkdir(exist_ok=True)               # 없으면 만들고, 있으면 그냥 지나간다

# 둘 다 True 가 나와야 정상 (False면 .env 설정 또는 data/ 폴더 확인)
print("OpenAI key:", bool(os.getenv("OPENAI_API_KEY")), "| PDF:", PDF.exists())

OpenAI key: True | PDF: True


In [3]:
# 필수 라이브러리가 전부 설치돼 있는지 확인한다
import importlib
from importlib.metadata import version

required = ["langchain", "langchain_openai", "langchain_community",
           "langchain_huggingface", "langchain_pymupdf4llm", "docling",
           "langgraph", "faiss", "rank_bm25", "sentence_transformers",
           "kiwipiepy", "fitz", "streamlit"]
# find_spec: import 를 실제로 하지 않고 "설치 여부"만 빠르게 확인하는 방법
missing = [m for m in required if importlib.util.find_spec(m) is None]
print("설치 안 된 것:", missing or "없음")

# 버전 확인 — langchain은 0.3.x, langgraph는 0.2.x 여야 한다.
# (langgraph 1.x가 섞이면 langchain 0.3.x와 충돌해 세션 5가 깨진다)
for pkg, expect in [("langchain", "0.3."), ("langgraph", "0.2."), ("streamlit", "1.58.")]:
    v = version(pkg)
    flag = "OK" if v.startswith(expect) else f"버전 확인 필요({expect}x)"
    print(f"{pkg} {v} - {flag}")

설치 안 된 것: 없음
langchain 0.3.27 - OK
langgraph 0.2.76 - OK
streamlit 1.58.0 - OK


In [4]:
import time
from langchain_huggingface import HuggingFaceEmbeddings

t0 = time.time()
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
)
print("임베딩 차원:", len(embeddings.embed_query("테스트")))
print("걸린 시간:", time.time() - t0)

임베딩 차원: 1024
걸린 시간: 7.156718969345093


In [5]:
# OpenAI LLM 을 한 번 호출해 API 키가 실제로 동작하는지 확인한다
from langchain_openai import ChatOpenAI

# temperature=0: 같은 질문에 최대한 같은 답 (수업 재현성을 위해 오늘 계속 0을 쓴다)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(llm.invoke("한 문장으로 자기소개 해줘.").content)

안녕하세요, 저는 다양한 주제에 대해 정보를 제공하고 대화를 나누는 AI입니다.


In [6]:
# 동봉된 캐시 파일이 모두 있는지 확인한다 — 오후 세션들이 이 파일을 로드해서 쓴다
for name in ["splits.pkl", "splits_all.pkl", "contextual_all.pkl", "faiss_index"]:
    print(name, "->", (CACHE / name).exists())

splits.pkl -> True
splits_all.pkl -> True
contextual_all.pkl -> True
faiss_index -> True


## 모델 미리 받아두기

`bge-m3`, `bge-reranker` 는 위의 셀이 실행되면서 이미 내려받았습니다.
남은 것은 **docling** — 첫 `convert()` 때 표 인식 모델을 내려받으므로,
아래 셀로 1쪽만 미리 변환해 다운로드를 끝내 둡니다. (첫 실행만 수 분, 이후엔 수 초)

강의 전 터미널에서 한 번에 받으려면:
```bash
poetry run python -c "from sentence_transformers import SentenceTransformer, CrossEncoder; \
SentenceTransformer('BAAI/bge-m3'); CrossEncoder('BAAI/bge-reranker-v2-m3')"
```

In [ ]:
# docling 준비 운동 — 첫 실행 때 표 인식 모델을 내려받게 만든다
import logging
from docling.document_converter import DocumentConverter

logging.disable(logging.INFO)   # OCR 진행 로그 숨김 (경고·오류는 표시됨)

# 1쪽만 변환 — 첫 실행이면 docling 모델을 내려받는다 (이후엔 수 초)
t0 = time.time()
result = DocumentConverter().convert(str(PDF), page_range=(1, 1))
print(f"docling OK — 1쪽 변환 {time.time()-t0:.1f}s")

docling OK — 1쪽 변환 605.9s


In [9]:
result.document.export_to_markdown()

"## 저출생 추세 반전을 위한 대책\n\n2024.  6.  19.\n\n## 저출산고령사회위원회\n\n관계부처 합동\n\n## 1. 일·가정 양립\n\n- ➊ 단기 육아휴직 도입 (年1 회 2 주 단위 육아휴직 허용 ) 18페이지\n- ➋ 육아휴직 급여 최대 상한 인상 (現 150→ 최대 250 만원 ) 19페이지 및 육아휴직 대체인력지원금 신설 및 지원금 확대 (現  80→120 만원 ) 23페이 지\n- ➌ 아빠 출산휴가 기간 (現  10 일 →20 일 , 근무일 기준 ), 청구기한 (現90→ 120 일 ) 및 분할횟수 확대 (現 1 회 →3 회 ) 21페이지\n- ➍ 출산휴가 · 육아휴직 통합신청제 도입 22페이지\n- ➎ 가족돌봄휴가 , 배우자출산휴가 등의 時 단위 사용 활성화 21페이지\n\n## 2. 교육·돌봄\n\n- ➊ 0∼5 세 단계적 무상교육 ‧ 보육 실현 ('25 년 5 세 → 임기 내 3, 4 세까지 확대 ) 29페이지\n- ➋ 늘봄 프로그램 단계적 무상운영 확대 ('2 5 년 초 1,2 →  '2 6 년 초 3 →  '27 년 초 4∼6) 31페이지\n- ➌ 틈새돌봄 확대 ( 시간제보육기관 확대 , 야간연장 · 휴일 · 방학운영 확대 ) 30페이지\n- ➍ 아이돌봄서비스 지원확대 , 외국인 가사관리사 활성화 등 가정돌봄 확충 31페이지\n- ➎ 대기업 · 지자체 등의 상생형 직장어린이집 확산 29페이지\n\n## 3대 분야 15대 핵심 과제"

## 직접 해보기

> 예시 코드를 **새 셀에 복사해** 실행해 보세요. (+ 버튼 또는 `B` 키로 셀 추가)

**1. `llm.invoke` 의 질문을 바꿔 다른 답을 받아 보세요.**
```python
print(llm.invoke("RAG가 무엇인지 한 문장으로 설명해줘.").content)
```

**2. 다른 문장을 임베딩해도 차원이 1024로 같은지 확인해 보세요.**
```python
vec = embeddings.embed_query("늘봄학교 프로그램")
print("차원:", len(vec), "| 앞 5개 값:", vec[:5])
```

**3. docling으로 표가 있는 32쪽을 변환해 보세요. (세션 2 예고편)**
```python
md = DocumentConverter().convert(str(PDF), page_range=(32, 32)).document.export_to_markdown()
print(md[:500])
```

In [12]:
print(llm.invoke("RAG가 무엇인지 한 문장으로 설명해줘.").content)

RAG는 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합하여 보다 정확하고 풍부한 응답을 생성하는 자연어 처리 기술입니다.


In [13]:
md = DocumentConverter().convert(str(PDF), page_range=(32, 32)).document.export_to_markdown()
print(md[:500])

| 구분      | 구분                                        | 현행                                                 | 개선                                                                        |
|-----------|---------------------------------------------|------------------------------------------------------|-----------------------------------------------------------------------------|
| 자격 요건 | 특별공급 공공 민간 ‣ 생애기간 중            | 특별공급 1회 당첨 제한                               | ‣ 신규출산가구는특별공급 재당첨 1회 허용          


## FAQ — 자주 막히는 지점

| 증상 | 해결 |
|------|------|
| `poetry: command not found` | 터미널/PowerShell을 닫고 새로 열어 재시도. 안 되면 STEP 2의 PATH(pipx ensurepath)를 다시 확인 |
| `python: command not found` (Windows) | Python 재설치 시 **Add Python to PATH** 체크 |
| `ModuleNotFoundError: No module named '...'` | `poetry install` 이 끝까지 안 됐을 가능성. 폴더에서 `poetry install` 재실행 |
| 위 "설치 안 된 것" 목록이 비어있지 않음 | 커널이 Poetry 가상환경이 아닐 수 있음 → STEP 5로 **Python (bcmd_rag_lecture)** 커널 다시 선택 |
| 커널이 안 보임 / 연결 안 됨 | STEP 5의 `ipykernel install` 재실행 후 VS Code 재시작 |
| 모델 다운로드가 너무 느림 | bge-m3·reranker가 각 수 GB. 정상이니 대기. 강의 전 "모델 미리 받아두기"를 한 번 실행 권장 |
| `OPENAI_API_KEY` 가 `False` | `.env` 파일이 **프로젝트 루트**에 있고 키가 올바른지 확인 (STEP 4) |

> 그래도 막히면 **오류 메시지 전체 + 본인 OS + 어느 단계**를 적어 Claude/ChatGPT/Gemini 에 물어보거나 강의 채널에 남겨주세요.

## 정리
- 위 셀이 전부 통과했다면 라이브러리 · 모델 · API 키 · 데이터가 모두 준비된 것입니다.
- 환경 설정의 8할은 모델 사전 다운로드입니다 — 강의 전에 받아 두면 이후 세션이 매끄럽습니다.
- 다음 세션에서는 RAG 개요를 잡고, 하루 종일 쓸 평가셋 12문항을 만듭니다.